# Paper 16 · CLIP

**Citation:** Alec Radford et al., “Learning Transferable Visual Models From Natural Language Supervision” (2021).

**Paper:** https://arxiv.org/abs/2103.00020

> **Scale gap:** We align two synthetic paired modalities generated from the same latent class, not web-scale image-text data.

## Mathematical Framework

Before reproducing the paper experimentally, work through:

- [Math 01 · Linear Algebra & Geometry](../../math/01_linear_algebra_geometry.ipynb)
- [Math 05 · Information Theory](../../math/05_information_theory.ipynb)
- [Math 06 · Optimization](../../math/06_optimization.ipynb)
- [Math 10 · Neural-Network Mathematics](../../math/10_neural_network_math.ipynb)

Your explanation should connect the paper's empirical claim to its **mathematical objective, representation, assumptions, and optimization/statistical argument**.

## Before you read
1. What makes a pair positive?
2. Why train both image-to-text and text-to-image directions?
3. How does a shared embedding space enable retrieval or zero-shot classification?

## Central claim
Contrastive learning on paired image-text data can align modalities in a shared embedding space and enable zero-shot transfer.

## Paired synthetic modalities

In [ ]:
import torch, numpy as np, matplotlib.pyplot as plt
from torch import nn
torch.manual_seed(0)
N,C=2000,6
label=torch.randint(0,C,(N,))
latent=nn.functional.one_hot(label,C).float()
A=latent@torch.randn(C,20)+.5*torch.randn(N,20)
B=latent@torch.randn(C,12)+.5*torch.randn(N,12)
tr=torch.arange(0,1600); te=torch.arange(1600,N)

## Dual encoders and symmetric contrastive loss

In [ ]:
img_enc=nn.Sequential(nn.Linear(20,32),nn.ReLU(),nn.Linear(32,16))
txt_enc=nn.Sequential(nn.Linear(12,32),nn.ReLU(),nn.Linear(32,16))
opt=torch.optim.Adam(list(img_enc.parameters())+list(txt_enc.parameters()),lr=.01)
def clip_loss(a,b,temp=.1):
    a=nn.functional.normalize(a,dim=1); b=nn.functional.normalize(b,dim=1)
    logits=a@b.T/temp; target=torch.arange(len(a))
    return (nn.functional.cross_entropy(logits,target)+nn.functional.cross_entropy(logits.T,target))/2
for step in range(400):
    ids=tr[torch.randint(0,len(tr),(256,))]
    loss=clip_loss(img_enc(A[ids]),txt_enc(B[ids]))
    opt.zero_grad(); loss.backward(); opt.step()
print("contrastive loss",float(loss))

## Cross-modal retrieval

In [ ]:
with torch.no_grad():
    ia=nn.functional.normalize(img_enc(A[te]),dim=1)
    tb=nn.functional.normalize(txt_enc(B[te]),dim=1)
    sim=ia@tb.T
    top=sim.argmax(1)
    pair_retrieval=(top==torch.arange(len(te))).float().mean().item()
    class_retrieval=(label[te][top]==label[te]).float().mean().item()
print("exact-pair top1",pair_retrieval,"same-class top1",class_retrieval)
plt.imshow(sim[:50,:50].numpy(),aspect="auto",cmap="viridis"); plt.colorbar(); plt.title("Cross-modal similarity matrix"); plt.show()

### Ablation
Shuffle pairings during training. What happens to retrieval? Explain why data pairing quality is part of the learning signal.

## Ablation table

| Variant | Metric / observation | What changed? | Why? |
|---|---:|---|---|
| Baseline |  |  |  |
| Ablation 1 |  |  |  |
| Ablation 2 |  |  |  |

## Defend the paper
1. What problem existed before this work?
2. What was actually new?
3. What evidence did this notebook reproduce?
4. What does the scale gap prevent you from claiming?
5. Which contribution remains important today?
6. What would you test next?